# Option D — Fully Offline Kaggle  *(JSON-parsed environments, behind firewall)*

This notebook runs **completely offline** on Kaggle (Internet toggle **OFF**) or
behind any corporate firewall — zero API calls are made to `three.arcprize.org`.

It synthesises learnings from Options A–C:
| Option | Mode | Learns |
|--------|------|--------|
| A | `normal` (local + API fallback) | Local environment scanning, dotenv pattern |
| B | `online` (API only) | Remote scorecard structure, API key secrets |
| C | `online`, full sweep | All-games iteration, Kaggle offline wheels |
| **D** | **`offline` (local only)** | **JSON-parsed envs, no HTTP, PMLL emergence** |

### Required Kaggle dataset inputs — attach before running
| Dataset slug | What it provides |
|---|---|
| `drQedwards/arc-agi-3-agents` | This repo (agents, main.py, pyproject.toml) |
| `drQedwards/arc-agi-3-environment-files` | `metadata.json` + game `.py` files |
| `drQedwards/pmll-memory-mcp-wheels` *(optional)* | Offline PMLL wheels |

> **No `ARC_API_KEY` required.**  Only `OPENAI_API_KEY` is needed (for LLM reasoning).

### How it works
1. Install `arc-agi`, `arcengine`, `openai`, `langchain` from PyPI wheels
2. Parse every `metadata.json` found in Kaggle inputs → populate `environment_files/`
3. Write `.env` with `OPERATION_MODE=offline` so `Arcade` never touches the network
4. Seed PMLL with a `stochastic_goose` identity (same as Options B/C baselines)
5. Run **ReasoningAgent** (baseline) — score captured even on failure
6. Run **WorldModelAgent** (emergence) — score captured even on failure
7. **Print two final scores — guaranteed output no matter what**
8. Flush PMLL silo; save JSON results artifact

## 1 — Install dependencies

In [ ]:
import subprocess, sys
from pathlib import Path

# ------------------------------------------------------------------
# Optional: install PMLL from pre-downloaded offline wheels
# (produced by arc_agi3_pmll_installer.ipynb)
# ------------------------------------------------------------------
PMLL_WHEELS = Path("/kaggle/input/pmll-memory-mcp-wheels/pmll_wheels")

def pip_install_offline(package: str, find_links: Path) -> None:
    """Install *package* from pre-downloaded wheels — no internet required."""
    if find_links.is_dir():
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", package,
            "--no-index", "--find-links", str(find_links), "--quiet",
        ])
        print(f"  [offline] installed {package} from {find_links}")
    else:
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", package, "--quiet",
        ])
        print(f"  [online]  installed {package} from PyPI")

pip_install_offline("pmll-memory-mcp", PMLL_WHEELS)

# Core dependencies (pre-installed in Kaggle Python image; update if missing)
REQUIRED = [
    "arc-agi>=0.9.1",
    "arcengine>=0.9.3",
    "openai==1.72.0",
    "langchain[openai]>=0.3.27",
    "langgraph>=0.6.3",
    "langgraph-checkpoint-sqlite>=2.0.11",
    "numpy>=2.3.2",
    "pillow>=11.2.1",
    "pydantic>=2.11.7",
    "smolagents>=1.20.0",
    "python-dotenv>=1.0.0",
    "requests>=2.32.4",
    "pandas>=2.0.0",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *REQUIRED])
print("All dependencies installed.")

## 2 — Load secrets and write `.env`

Add your `OPENAI_API_KEY` via **Add-ons → Secrets** in the Kaggle editor.  
`ARC_API_KEY` is **not required** for Option D — no API calls are made.

In [ ]:
import os
from pathlib import Path

def get_secret(name: str, required: bool = False) -> str:
    """Read a Kaggle secret or fall back to env var."""
    try:
        from kaggle_secrets import UserSecretsClient
        val = UserSecretsClient().get_secret(name)
        if val:
            return val
    except Exception:
        pass
    val = os.environ.get(name, "")
    if required and not val:
        raise RuntimeError(
            f"Secret '{name}' not found. Add it via Add-ons → Secrets."
        )
    return val

openai_api_key = get_secret("OPENAI_API_KEY", required=True)
agentops_key   = get_secret("AGENTOPS_API_KEY")   # optional

REPO_DIR  = Path("/kaggle/working/ARC-AGI-3-Agents")
ENV_DIR   = REPO_DIR / "environment_files"

REPO_DIR.mkdir(parents=True, exist_ok=True)

# OPERATION_MODE=offline → Arcade never makes HTTP requests
env_text = f"""# Auto-generated by arc_agi3_option_d_offline_kaggle.ipynb
DEBUG=False
RECORDINGS_DIR=recordings
OPERATION_MODE=offline
ENVIRONMENTS_DIR=environment_files
WDM_LOG=0
OPENAI_API_KEY={openai_api_key}
ARC_API_KEY=
AGENTOPS_API_KEY={agentops_key}
"""

(REPO_DIR / ".env").write_text(env_text)

# Also export into this process
os.environ["OPENAI_API_KEY"]  = openai_api_key
os.environ["OPERATION_MODE"]  = "offline"
os.environ["ENVIRONMENTS_DIR"] = str(ENV_DIR)
os.environ["PYTHONPATH"]       = str(REPO_DIR)

print("OPENAI_API_KEY  set:", bool(openai_api_key))
print("OPERATION_MODE  :", os.environ["OPERATION_MODE"])
print(".env written to :", REPO_DIR / ".env")

## 3 — Set up the repo

The repo is read from the attached `arc-agi-3-agents` dataset input.  
If that dataset is not attached, the cell falls back to cloning from GitHub
(requires Internet toggle ON).

In [ ]:
import shutil

REPO_INPUT = Path("/kaggle/input/arc-agi-3-agents")
REPO_CLONE = REPO_DIR   # already set above

if REPO_INPUT.is_dir():
    if not REPO_CLONE.exists():
        shutil.copytree(REPO_INPUT, REPO_CLONE)
    print(f"Repo sourced from dataset input → {REPO_CLONE}")
elif not REPO_CLONE.is_dir():
    # Requires Internet — only runs if no dataset input is attached
    subprocess.check_call([
        "git", "clone", "--depth=1",
        "https://github.com/drQedwards/ARC-AGI-3-Agents.git",
        str(REPO_CLONE),
    ])
    print(f"Repo cloned from GitHub → {REPO_CLONE}")
else:
    print(f"Repo already present at {REPO_CLONE}")

os.chdir(REPO_CLONE)
print("Working directory:", Path.cwd())

## 4 — Parse Kaggle JSON dataset → `environment_files/`

This is the key step that makes Option D offline.  
Every `metadata.json` found under `/kaggle/input/` is treated as an `EnvironmentInfo`
descriptor.  The sibling `.py` file (the game implementation) is copied alongside it
into `environment_files/{game_id}/` so `Arcade(OPERATION_MODE=offline)` can load it
without touching the network.

If no environments are found the agents still run and return `levels_completed = 0`,
preserving the **two-score guarantee**.

In [ ]:
import json

ENV_DIR = REPO_CLONE / "environment_files"
ENV_DIR.mkdir(parents=True, exist_ok=True)

# Update the env var to point at the working-directory copy
os.environ["ENVIRONMENTS_DIR"] = str(ENV_DIR)

# Search roots: all Kaggle dataset inputs + any envs already in the repo
SEARCH_ROOTS = [
    Path("/kaggle/input"),
    REPO_CLONE / "environment_files",
]

installed: list[str] = []
skipped:   list[str] = []

for root in SEARCH_ROOTS:
    if not root.exists():
        continue
    for meta_file in sorted(root.rglob("metadata.json")):
        # Skip files already inside ENV_DIR (avoid double-copy)
        try:
            meta_file.relative_to(ENV_DIR)
            continue
        except ValueError:
            pass

        try:
            meta = json.loads(meta_file.read_text(encoding="utf-8"))
            game_id = meta.get("game_id", "").strip()
            if not game_id:
                skipped.append(str(meta_file))
                continue

            dst_dir = ENV_DIR / game_id

            if dst_dir.exists():
                # Already installed from an earlier iteration or prior run
                if game_id not in installed:
                    installed.append(game_id)
                continue

            dst_dir.mkdir(parents=True, exist_ok=True)
            shutil.copy2(meta_file, dst_dir / "metadata.json")

            # Copy all Python game files in the same directory
            py_files = list(meta_file.parent.glob("*.py"))
            for py_file in py_files:
                shutil.copy2(py_file, dst_dir / py_file.name)

            installed.append(game_id)
            py_names = [p.name for p in py_files]
            print(f"  ✓ {game_id:20s} ← {meta_file.parent.name}/  (py: {py_names})")

        except Exception as exc:
            skipped.append(f"{meta_file} ({exc})")

print()
print(f"Environments installed : {len(installed)}  → {installed}")
if skipped:
    print(f"Skipped (no game_id)   : {len(skipped)}")

if not installed:
    print()
    print("╔══════════════════════════════════════════════════════╗")
    print("║  WARNING: No environments found in Kaggle inputs.   ║")
    print("║  Both agents will run and return levels_completed=0 ║")
    print("║  (the two-score guarantee is still met).            ║")
    print("║  Attach 'arc-agi-3-environment-files' dataset to    ║")
    print("║  get real scores.                                   ║")
    print("╚══════════════════════════════════════════════════════╝")

GAMES_AVAILABLE = installed[:]

## 5 — Seed PMLL memory: Stochastic Goose identity

We boot a PMLL session and store an initial identity entry `stochastic_goose`.
After the WorldModelAgent run, the entry is overwritten with `world_model_agent`
data — the **emergence** transition from stochastic baseline to world-model-guided
behaviour (Option D continuation of the Option C offline pattern).

In [ ]:
import importlib.util, time

PMLL_AVAILABLE = importlib.util.find_spec("pmll_memory_mcp") is not None

class _PMLLShim:
    """Thin in-process shim used when pmll_memory_mcp is absent.

    Provides the same surface as the real client so the rest of the notebook
    is agnostic to whether PMLL is installed.
    """

    def __init__(self) -> None:
        self._store: dict = {}
        self._graph: list = []
        print("  [shim] pmll_memory_mcp not found — using in-process shim")

    def init(self, session_id: str, silo_size: int = 256) -> dict:
        self._store.setdefault(session_id, {})
        return {"status": "ok", "session_id": session_id, "silo_size": silo_size}

    def set(self, session_id: str, key: str, value) -> dict:
        self._store.setdefault(session_id, {})[key] = value
        return {"status": "stored", "key": key}

    def peek(self, session_id: str, key: str):
        return self._store.get(session_id, {}).get(key)

    def upsert_memory_node(
        self,
        session_id: str,
        node_type: str,
        label: str,
        content: str,
        metadata: dict | None = None,
    ) -> dict:
        for node in self._graph:
            if node["label"] == label:
                node.update({"type": node_type, "content": content,
                              "metadata": metadata or {},
                              "updated_at": time.time()})
                return {"status": "updated", "label": label}
        self._graph.append({"type": node_type, "label": label,
                             "content": content, "metadata": metadata or {},
                             "created_at": time.time()})
        return {"status": "created", "label": label}

    def search_memory_graph(self, session_id: str, query: str, top_k: int = 5) -> dict:
        hits = [n for n in self._graph if query.lower() in n["content"].lower()]
        return {"results": hits[:top_k]}

    def dump_graph(self, session_id: str) -> list:
        return list(self._graph)

    def flush(self, session_id: str) -> dict:
        self._store.pop(session_id, None)
        return {"status": "flushed"}


if PMLL_AVAILABLE:
    try:
        import pmll_memory_mcp as pmll_mod
        pmll = pmll_mod
        print("  [pmll] using pmll_memory_mcp package")
    except Exception:
        pmll = _PMLLShim()
else:
    pmll = _PMLLShim()

SESSION_ID = "arc3_option_d_offline"
pmll.init(SESSION_ID, silo_size=512)

# Seed: stochastic_goose baseline identity (same as Option C offline)
pmll.set(SESSION_ID, "agent_identity",  "stochastic_goose")
pmll.set(SESSION_ID, "run_mode",        "offline_option_d")
pmll.set(SESSION_ID, "games_available", json.dumps(GAMES_AVAILABLE))
pmll.upsert_memory_node(
    SESSION_ID, "identity", "stochastic_goose",
    "Baseline stochastic goose identity seeded at Option D start. "
    "Will be superseded by world_model_agent after WorldModelAgent run.",
)

print(f"PMLL session '{SESSION_ID}' initialised.")
print("Identity:", pmll.peek(SESSION_ID, "agent_identity"))


## 6 — Helper: run agent and capture score

The helper **always** returns a result dict with `levels_completed` (default `0`).
Exceptions, timeouts, and zero-environment runs are all caught and logged —
they never prevent the two-score final output from being printed.

In [ ]:
import re, time


def run_agent(
    agent_name: str,
    tags: str,
    game: str | None = None,
    timeout: int = 1800,
    pmll_session: str = SESSION_ID,
) -> dict:
    """Run *agent_name* via main.py and ALWAYS return a scored result dict.

    The returned dict always contains:
      agent, tags, game, levels_completed (≥0), win_levels (≥0),
      elapsed_s, returncode, error (None on success), log_tail.
    """
    result: dict = {
        "agent":            agent_name,
        "tags":             tags,
        "game":             game or "ALL",
        "levels_completed": 0,   # guaranteed default
        "win_levels":       0,
        "elapsed_s":        0.0,
        "returncode":       -1,
        "error":            None,
        "log_tail":         "",
    }

    try:
        cmd = [
            sys.executable,
            str(REPO_CLONE / "main.py"),
            f"--agent={agent_name}",
            f"--tags={tags}",
        ]
        if game:
            cmd.append(f"--game={game}")

        env = os.environ.copy()
        env["PYTHONPATH"]       = str(REPO_CLONE)
        env["OPERATION_MODE"]   = "offline"
        env["ENVIRONMENTS_DIR"] = str(ENV_DIR)
        env["DOTENV_PATH"]      = str(REPO_CLONE / ".env")
        env["TESTING"]          = "False"

        print(f"\n>>> Starting {agent_name!r} (tags={tags!r}, game={game or 'ALL'})")
        t0 = time.time()

        proc = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            timeout=timeout,
            env=env,
        )

        result["elapsed_s"]  = round(time.time() - t0, 1)
        result["returncode"] = proc.returncode

        full_log = proc.stdout + proc.stderr
        result["log_tail"] = full_log[-4000:]

        # ── Parse FINAL SCORECARD REPORT JSON block ──────────────────────
        json_match = re.search(
            r"FINAL SCORECARD REPORT.*?(\{.*?\})",
            full_log, re.DOTALL,
        )
        if json_match:
            try:
                summary = json.loads(json_match.group(1))
                result["levels_completed"] = int(summary.get("levels_completed", 0))
                result["win_levels"]       = int(summary.get("win_levels", 0))
            except (json.JSONDecodeError, ValueError):
                pass

        # ── Fallback: scan log lines for numeric level completions ────────
        for line in full_log.splitlines():
            m = re.search(r'"levels_completed"\s*:\s*(\d+)', line)
            if m:
                result["levels_completed"] = max(
                    result["levels_completed"], int(m.group(1))
                )
            m = re.search(r'"win_levels"\s*:\s*(\d+)', line)
            if m:
                result["win_levels"] = max(result["win_levels"], int(m.group(1)))

        # ── Inline log line: "levels completed N" ────────────────────────
        for line in full_log.splitlines():
            m = re.search(r'levels completed (\d+)', line, re.IGNORECASE)
            if m:
                result["levels_completed"] = max(
                    result["levels_completed"], int(m.group(1))
                )

    except subprocess.TimeoutExpired:
        result["error"] = f"Timeout after {timeout}s"
    except Exception as exc:
        result["error"] = str(exc)

    # ── PMLL: upsert per-agent memory node (emergence step) ──────────────
    try:
        snapshot = json.dumps({
            "agent":            agent_name,
            "game":             result["game"],
            "levels_completed": result["levels_completed"],
            "win_levels":       result["win_levels"],
            "elapsed_s":        result["elapsed_s"],
            "error":            result["error"],
        })
        pmll.upsert_memory_node(
            pmll_session, "run_result", f"optionD_{agent_name}", snapshot
        )
        if agent_name == "worldmodelagent":
            # Emergence: overwrite the stochastic_goose identity
            pmll.set(pmll_session, "agent_identity", "world_model_agent")
            pmll.upsert_memory_node(
                pmll_session, "identity", "world_model_agent",
                f"Emerged from stochastic_goose after WorldModelAgent run. "
                f"levels_completed={result['levels_completed']}",
            )
    except Exception:
        pass  # PMLL failure must never suppress the score

    status = "OK" if result["error"] is None else f"ERR({result['error'][:60]})"
    print(
        f"    {agent_name}: levels_completed={result['levels_completed']}, "
        f"win_levels={result['win_levels']}, "
        f"elapsed={result['elapsed_s']}s, status={status}"
    )
    return result


print("run_agent helper defined.")

## 7 — Run ReasoningAgent (baseline — Stochastic Goose identity)

In [ ]:
print("Pre-run PMLL identity:", pmll.peek(SESSION_ID, "agent_identity"))

result_baseline = run_agent(
    agent_name="reasoningagent",
    tags="baseline,optionD,offline,stochastic_goose",
)

print(json.dumps(
    {k: v for k, v in result_baseline.items() if k != "log_tail"},
    indent=2,
))

## 8 — Run WorldModelAgent (emergence: stochastic goose → world model)

In [ ]:
result_worldmodel = run_agent(
    agent_name="worldmodelagent",
    tags="worldmodel,optionD,offline,emergence",
)

print(json.dumps(
    {k: v for k, v in result_worldmodel.items() if k != "log_tail"},
    indent=2,
))

print("\nPost-run PMLL identity:", pmll.peek(SESSION_ID, "agent_identity"))

## 9 — OPTION D FINAL SCORES

Two scores are printed here **no matter what** — even if an agent errored,
timed out, or found no environments.  A score of `0` indicates the agent
could not complete any levels.

In [ ]:
import pandas as pd

# ── Guaranteed extraction — always an int, never raises ──────────────────
lc_baseline   = int(result_baseline.get("levels_completed", 0))
lc_worldmodel = int(result_worldmodel.get("levels_completed", 0))
wl_baseline   = int(result_baseline.get("win_levels",       0))
wl_worldmodel = int(result_worldmodel.get("win_levels",     0))

# ── Table ─────────────────────────────────────────────────────────────────
rows = [
    {
        "agent":            "ReasoningAgent  (baseline)",
        "mode":             "offline / Option D",
        "levels_completed": lc_baseline,
        "win_levels":       wl_baseline,
        "elapsed_s":        result_baseline["elapsed_s"],
        "error":            result_baseline["error"] or "—",
    },
    {
        "agent":            "WorldModelAgent (world model)",
        "mode":             "offline / Option D",
        "levels_completed": lc_worldmodel,
        "win_levels":       wl_worldmodel,
        "elapsed_s":        result_worldmodel["elapsed_s"],
        "error":            result_worldmodel["error"] or "—",
    },
]
df = pd.DataFrame(rows).set_index("agent")

print()
print("=" * 62)
print("  OPTION D — FINAL SCORES  (offline, behind firewall)")
print("=" * 62)
print(df.to_string())
print("=" * 62)
print()
print(f"  Baseline   (ReasoningAgent):   levels_completed = {lc_baseline:>4}  |  win_levels = {wl_baseline}")
print(f"  WorldModel (WorldModelAgent):  levels_completed = {lc_worldmodel:>4}  |  win_levels = {wl_worldmodel}")
print("=" * 62)

# Warn about errors but do NOT raise — scores are already printed above
if result_baseline["error"]:
    print(f"  [WARN] Baseline error  : {result_baseline['error']}")
if result_worldmodel["error"]:
    print(f"  [WARN] WorldModel error: {result_worldmodel['error']}")
if not GAMES_AVAILABLE:
    print("  [WARN] No game environments were found in Kaggle inputs.")
    print("         Attach the 'arc-agi-3-environment-files' dataset for real scores.")

## 10 — PMLL memory graph after emergence

In [ ]:
print("=== Memory graph after Option D emergence ===")
try:
    graph = pmll.dump_graph(SESSION_ID) if hasattr(pmll, "dump_graph") else []
    for node in graph:
        print(f"  [{node.get('type', '?')}] {node.get('label', '?')}: "
              f"{str(node.get('content', ''))[:120]}")
except Exception as exc:
    print(f"  (PMLL graph unavailable: {exc})")

print()
print("=== Search: 'optionD' ===")
try:
    hits = pmll.search_memory_graph(SESSION_ID, "optionD")
    for h in hits.get("results", []):
        print(f"  {h.get('label', '?')}: {str(h.get('content', ''))[:120]}")
except Exception:
    pass

print()
print("=== Final agent identity ===")
print(" ", pmll.peek(SESSION_ID, "agent_identity"))

## 11 — Save results and flush PMLL session

In [ ]:
from datetime import datetime

timestamp = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
out_path  = Path("/kaggle/working") / f"option_d_results_{timestamp}.json"

try:
    memory_graph = pmll.dump_graph(SESSION_ID) if hasattr(pmll, "dump_graph") else []
except Exception:
    memory_graph = []

results = {
    "timestamp":          timestamp,
    "option":             "D_offline",
    "pmll_session_id":    SESSION_ID,
    "games_available":    GAMES_AVAILABLE,
    "final_scores": {
        "baseline_levels_completed":    lc_baseline,
        "baseline_win_levels":          wl_baseline,
        "worldmodel_levels_completed":  lc_worldmodel,
        "worldmodel_win_levels":        wl_worldmodel,
    },
    "memory_graph": memory_graph,
    "runs": [
        {k: v for k, v in result_baseline.items()   if k != "log_tail"},
        {k: v for k, v in result_worldmodel.items() if k != "log_tail"},
    ],
}

out_path.write_text(json.dumps(results, indent=2, default=str))
print(f"Results saved → {out_path}")

# Save per-agent logs
for r in [result_baseline, result_worldmodel]:
    log_path = Path("/kaggle/working") / f"{r['agent']}_optionD_{timestamp}.log"
    log_path.write_text(r.get("log_tail", ""))
    print(f"Log  saved  → {log_path}")

# Flush PMLL short-term silo (long-term graph persists in the MCP store)
try:
    pmll.flush(SESSION_ID)
    print("PMLL silo flushed.")
except Exception:
    pass

print()
print("Option D complete.")
print(f"  Baseline   levels_completed : {lc_baseline}")
print(f"  WorldModel levels_completed : {lc_worldmodel}")